# Merge BERTopic Models Across Outlets

This notebook is the canonical merged-topic workflow.

It does four things in order:
1. load the seven saved **outlet-specific** BERTopic models from `1a_BERTopic/local_outputs/`
2. merge those outlet models into one shared topic space
3. assign the merged topics back onto the canonical combined corpus in `00_Initial EDA/df_combined.csv`
4. review the merged topics via topic tables, a 3D UMAP, and keyword-comparison outputs

Important lineage:
- these seven models are **not** the legacy overall model from `00_Initial EDA/07_OverallTM.ipynb`
- they were trained separately in the outlet notebooks and saved individually
- the cleaned outlet corpora behind those models are the same source material that was later concatenated into `df_combined.csv`
- therefore the merged workflow now combines outlet-specific topic spaces with the canonical combined article corpus


In [ ]:
import os
import sys
from pathlib import Path

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MIN_SIMILARITY = 0.7


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing .git")


PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_ROOT = PROJECT_ROOT / "1a_BERTopic"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")

MODEL_DIR_CANDIDATES = [PROJECT_ROOT / "1a_BERTopic" / "local_outputs"]
MERGED_SAVE_DIR = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_all_outlets_model"

print(f"Project root: {PROJECT_ROOT}")
print("Model path candidates:")
for candidate in MODEL_DIR_CANDIDATES:
    print(f"  - {candidate}")


## 1. Load Saved Outlet Models

This cell loads the seven saved **outlet-specific** BERTopic models from `1a_BERTopic/local_outputs/`.

These models are **not** taken from the legacy overall topic model in `00_Initial EDA/07_OverallTM.ipynb`.
They come from the separate outlet notebooks in `00_Initial EDA/`, trained on the cleaned outlet corpora that were later concatenated into `df_combined.csv`.

So the logic is:
- per-outlet notebooks train one BERTopic model per outlet
- this notebook merges those seven topic spaces
- article-level assignment is then rerun on `df_combined.csv` so the final merged dataset stays on the canonical thesis corpus


In [ ]:
import importlib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from bertopic import BERTopic

import merged_outlets_analysis as moa
moa = importlib.reload(moa)

OUTLET_SPECS = moa.OUTLET_SPECS
apply_topic_name_overrides = moa.apply_topic_name_overrides
build_article_topic_dataset = moa.build_article_topic_dataset
build_merged_article_frame = moa.build_merged_article_frame
build_merged_article_umap_3d = moa.build_merged_article_umap_3d
build_topic_keyword_summary = moa.build_topic_keyword_summary
build_topic_keyword_rank_table = moa.build_topic_keyword_rank_table
refresh_topic_representations = moa.refresh_topic_representations
combine_prepared_documents = moa.combine_prepared_documents
export_article_topic_dataset = moa.export_article_topic_dataset
export_df_combined_with_topics = moa.export_df_combined_with_topics
load_topic_name_overrides = moa.load_topic_name_overrides
plot_merged_topic_umap_3d = moa.plot_merged_topic_umap_3d
resolve_model_paths = moa.resolve_model_paths

MODEL_PATHS = resolve_model_paths(MODEL_DIR_CANDIDATES)
loaded_models = {
    key: BERTopic.load(model_path, embedding_model=EMBEDDING_MODEL)
    for key, model_path in MODEL_PATHS.items()
}

for key, model in loaded_models.items():
    topic_info = model.get_topic_info()
    print(f"Loaded {key}: {MODEL_PATHS[key]} ({len(topic_info)} rows in topic info)")

models_to_merge = [
    loaded_models["tagesschau"],
    loaded_models["rt"],
    loaded_models["antispiegel"],
    loaded_models["tichys"],
    loaded_models["nius"],
    loaded_models["compact"],
    loaded_models["deutschlandkurier"],
]


## 2. Merge The Topic Models

This creates the shared merged-topic space that all downstream analysis uses.


In [ ]:
import shutil

merged_model = BERTopic.merge_models(models_to_merge, min_similarity=MIN_SIMILARITY)
merged_topic_info_display = merged_model.get_topic_info().copy()

display(merged_topic_info_display.head(10))
print("Merged topic count (including outlier topic -1):", len(merged_topic_info_display))

SAVE_MERGED_MODEL = False
if SAVE_MERGED_MODEL:
    if MERGED_SAVE_DIR.exists():
        shutil.rmtree(MERGED_SAVE_DIR)
    merged_model.save(
        MERGED_SAVE_DIR,
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=EMBEDDING_MODEL,
    )
    print(f"Saved merged model to {MERGED_SAVE_DIR}")
else:
    print(f"Skipping save. Set SAVE_MERGED_MODEL = True to write {MERGED_SAVE_DIR}")


## 3. Assign Topics From `df_combined`, Apply Outlier Reduction, And Export Clean Dataframes

This section treats `00_Initial EDA/df_combined.csv` as the canonical article corpus.

That means:
- outlet-specific raw cleaning such as the earlier Compact cleanup is assumed to already be baked into `df_combined`
- this notebook does **not** rebuild article rows from the raw outlet loaders
- the only additional filtering comes from the BERTopic preparation step for each outlet (`min_text_chars`, `min_tokens`, cleaned-document deduplication, boilerplate removal)
- after topic assignment, this notebook applies outlier reduction to the merged assignments before exporting the final article-topic tables

Exports written here:
- `1a_BERTopic/local_outputs/merged_articles_with_topics.csv`
- `00_Initial EDA/df_combined_with_topics.csv`
- `1a_BERTopic/local_outputs/merged_topics_overview.csv`
- `1a_BERTopic/local_outputs/merged_top_topics_summary.csv`
- `1a_BERTopic/local_outputs/merged_top_topics_top20_keywords_long.csv`
- `1a_BERTopic/local_outputs/merged_top_topics_top20_keywords_comparison.csv`

`row_id` now comes directly from `df_combined`, so topic assignments map back exactly to the canonical corpus.


In [ ]:
load_df_combined = moa.load_df_combined
load_all_prepared_documents_from_df_combined = moa.load_all_prepared_documents_from_df_combined
split_df_combined_by_outlet = moa.split_df_combined_by_outlet

APPLY_OUTLIER_REDUCTION = True
OUTLIER_STRATEGY = "c-tf-idf"
OUTLIER_THRESHOLD = 0.10

df_combined = load_df_combined(PROJECT_ROOT)
df_combined_by_outlet = split_df_combined_by_outlet(PROJECT_ROOT)
prepared_by_outlet = load_all_prepared_documents_from_df_combined(PROJECT_ROOT)

prepared_summary = pd.DataFrame(
    [
        {
            "Outlet": OUTLET_SPECS[key].label,
            "df_combined_rows": len(df_combined_by_outlet[key]),
            "Prepared_Documents": len(df),
            "Dropped_In_Preparation": len(df_combined_by_outlet[key]) - len(df),
        }
        for key, df in prepared_by_outlet.items()
    ]
).sort_values("Outlet").reset_index(drop=True)
display(prepared_summary)

combined_prepared = combine_prepared_documents(prepared_by_outlet)
print("df_combined rows:", len(df_combined))
print("Combined prepared documents from df_combined:", len(combined_prepared))
print("Documents dropped during BERTopic preparation:", len(df_combined) - len(combined_prepared))
print(f"Outlier reduction: {APPLY_OUTLIER_REDUCTION} ({OUTLIER_STRATEGY}, threshold={OUTLIER_THRESHOLD})")

merged_articles, merged_topic_info_display, merged_umap_model = build_merged_article_frame(
    merged_model,
    combined_prepared,
    apply_outlier_reduction=APPLY_OUTLIER_REDUCTION,
    outlier_strategy=OUTLIER_STRATEGY,
    outlier_threshold=OUTLIER_THRESHOLD,
)
print("Final outlier articles after reduction:", int((merged_articles["merged_topic"] == -1).sum()))

topic_name_overrides = load_topic_name_overrides(PROJECT_ROOT)
merged_topic_info_named = apply_topic_name_overrides(merged_topic_info_display, topic_name_overrides)
article_topic_df = build_article_topic_dataset(
    merged_articles,
    merged_topic_info_display,
    topic_name_overrides=topic_name_overrides,
)
article_topic_export_paths = export_article_topic_dataset(PROJECT_ROOT, article_topic_df)
df_combined_with_topics_df, df_combined_export_paths = export_df_combined_with_topics(
    PROJECT_ROOT,
    article_topic_df,
)

display(
    article_topic_df[
        [
            "row_id",
            "outlet_label",
            "document_id",
            "merged_topic",
            "topic_display_label",
        ]
    ].head(10)
)

display(
    df_combined_with_topics_df["topic_match_status"]
    .value_counts(dropna=False)
    .rename_axis("topic_match_status")
    .to_frame("rows")
)

print("Saved article-topic dataset:")
for file_type, export_path in article_topic_export_paths.items():
    print(f"  {file_type}: {export_path}")

print("Saved df_combined-with-topics dataset:")
for file_type, export_path in df_combined_export_paths.items():
    print(f"  {file_type}: {export_path}")


## 4. List All Final Merged Topics

This table lists all substantive merged topics after the final outlier-reduced assignment.
It keeps both the final assigned article count and the original model-side count for reference.


In [ ]:
TOPIC_LIST_EXPORT_PATH = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_topics_overview.csv"

all_topics_df = (
    merged_topic_info_named.loc[
        merged_topic_info_named["Topic"] != -1,
        [
            "DisplayTopic",
            "Topic",
            "AssignedCount",
            "AssignedShare",
            "ModelCount",
            "DisplayLabel",
            "topic_label",
            "Name",
        ],
    ]
    .rename(
        columns={
            "AssignedCount": "AssignedArticleCount",
            "AssignedShare": "AssignedArticleShare",
            "ModelCount": "OriginalModelCount",
        }
    )
    .sort_values("DisplayTopic")
    .reset_index(drop=True)
)

all_topics_df.to_csv(TOPIC_LIST_EXPORT_PATH, index=False)
display(all_topics_df)
print(f"Substantive merged topics: {len(all_topics_df)}")
print(f"Saved full topic list to: {TOPIC_LIST_EXPORT_PATH}")


## 5. 3D UMAP Of The Merged Topic Space

This projects the article embeddings into a 3D UMAP so the shared topic space can be explored interactively.


In [ ]:
UMAP_3D_TOP_N = 20
MAX_POINTS_3D = 12000

merged_articles_3d = build_merged_article_umap_3d(merged_model, article_topic_df)
umap_3d_fig = plot_merged_topic_umap_3d(
    merged_articles_3d,
    merged_topic_info_named,
    top_n=UMAP_3D_TOP_N,
    max_points=MAX_POINTS_3D,
)
umap_3d_fig


## 6. Keyword Summary And Comparison View

This section ranks topics by the final assigned article counts after outlier reduction, then exports:
- a top-topic summary table
- a long table with one row per `(topic, keyword)` pair
- a side-by-side comparison table for the available keywords per topic

Important: the current saved merged-topic representation stores `10` keywords per topic.
So `TOP_K_WORDS = 20` is the requested maximum, but the current artifact can only return up to `10` unless the model is rebuilt/saved with a longer representation.


In [ ]:
REFRESH_TOPIC_REPRESENTATIONS = True
REFRESH_TOP_N_WORDS = 20
TOP_N_TOPICS = 10
TOP_K_WORDS = 20
ALL_TOPICS_K_WORDS = 20
TOPIC_FILTER = None          # Example: "russ" or "migration"
TOPIC_IDS_TO_COMPARE = []    # Example: [3, 7, 12]

TOPIC_LIST_EXPORT_PATH = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_topics_overview.csv"
REFRESHED_TOPIC_LIST_EXPORT_PATH = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_topics_overview_refreshed.csv"
TOP_TOPIC_SUMMARY_EXPORT_PATH = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_top_topics_summary.csv"
TOP_TOPIC_KEYWORDS_EXPORT_PATH = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_top_topics_top20_keywords_long.csv"
TOP_TOPIC_COMPARISON_EXPORT_PATH = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_top_topics_top20_keywords_comparison.csv"
TOP_TOPIC_RANK_TABLE_EXPORT_PATH = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_top_topics_top20_keywords_rank_table.csv"
ALL_TOPIC_KEYWORDS_EXPORT_PATH = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_all_topics_top20_keywords_long.csv"
ALL_TOPIC_RANK_TABLE_EXPORT_PATH = PROJECT_ROOT / "1a_BERTopic" / "local_outputs" / "merged_all_topics_top20_keywords_rank_table.csv"

canonical_topic_info_display = moa.attach_assigned_counts_to_topic_info(
    merged_topic_info_display,
    article_topic_df["merged_topic"].astype(int),
    total_docs=len(article_topic_df),
)
canonical_topic_info_named = apply_topic_name_overrides(
    canonical_topic_info_display,
    topic_name_overrides,
)
canonical_topic_info_named.to_csv(TOPIC_LIST_EXPORT_PATH, index=False)

keyword_model = merged_model
if REFRESH_TOPIC_REPRESENTATIONS:
    keyword_model, refreshed_topic_info_display = refresh_topic_representations(
        merged_model,
        article_topic_df,
        top_n_words=REFRESH_TOP_N_WORDS,
        config=moa.BERTopicConfig(top_n_words=REFRESH_TOP_N_WORDS),
    )
    refreshed_topic_info_named = apply_topic_name_overrides(
        refreshed_topic_info_display,
        topic_name_overrides,
    )
    refreshed_topic_info_named.to_csv(REFRESHED_TOPIC_LIST_EXPORT_PATH, index=False)
else:
    refreshed_topic_info_named = canonical_topic_info_named.copy()


top_topic_size_df, topic_keyword_weights_df, topic_keyword_comparison_df = build_topic_keyword_summary(
    keyword_model,
    canonical_topic_info_named,
    top_n_topics=TOP_N_TOPICS,
    top_k_words=TOP_K_WORDS,
    topic_ids=TOPIC_IDS_TO_COMPARE or None,
    topic_filter=TOPIC_FILTER,
)
top_topic_rank_table_df = build_topic_keyword_rank_table(topic_keyword_weights_df)

all_topic_size_df, all_topic_keyword_weights_df, _ = build_topic_keyword_summary(
    keyword_model,
    canonical_topic_info_named,
    top_n_topics=None,
    top_k_words=ALL_TOPICS_K_WORDS,
)
all_topic_rank_table_df = build_topic_keyword_rank_table(all_topic_keyword_weights_df)

keyword_source_label = "refreshed_assigned_corpus" if REFRESH_TOPIC_REPRESENTATIONS else "saved_model"
for export_df in (topic_keyword_weights_df, all_topic_keyword_weights_df):
    if not export_df.empty:
        export_df["keyword_source"] = keyword_source_label

top_topic_size_df.to_csv(TOP_TOPIC_SUMMARY_EXPORT_PATH, index=False)
topic_keyword_weights_df.to_csv(TOP_TOPIC_KEYWORDS_EXPORT_PATH, index=False)
topic_keyword_comparison_df.to_csv(TOP_TOPIC_COMPARISON_EXPORT_PATH, index=False)
top_topic_rank_table_df.to_csv(TOP_TOPIC_RANK_TABLE_EXPORT_PATH, index=False)
all_topic_keyword_weights_df.to_csv(ALL_TOPIC_KEYWORDS_EXPORT_PATH, index=False)
all_topic_rank_table_df.to_csv(ALL_TOPIC_RANK_TABLE_EXPORT_PATH, index=False)

display(canonical_topic_info_named.head(10))
display(top_topic_size_df)
display(topic_keyword_comparison_df)
display(top_topic_rank_table_df)
display(all_topic_rank_table_df)

print(
    f"Canonical topic overview saved to: {TOPIC_LIST_EXPORT_PATH}"
)
if REFRESH_TOPIC_REPRESENTATIONS:
    print(
        f"Refreshed topic representation overview saved to: {REFRESHED_TOPIC_LIST_EXPORT_PATH}"
    )
print(
    f"Keyword tables use source={keyword_source_label} while keeping canonical topic labels from the saved merged model."
)
print(
    f"Built keyword summaries for {len(top_topic_size_df)} selected topics and {len(all_topic_size_df)} substantive topics in total."
)
print("keyword_weight is a c-TF-IDF distinctiveness score, not a document probability.")
print("The all-topic rank table gives one row per topic with keyword_01 ... keyword_20 for direct comparison.")
print(f"Saved selected-topic summary to: {TOP_TOPIC_SUMMARY_EXPORT_PATH}")
print(f"Saved selected-topic long keyword table to: {TOP_TOPIC_KEYWORDS_EXPORT_PATH}")
print(f"Saved selected-topic comparison table to: {TOP_TOPIC_COMPARISON_EXPORT_PATH}")
print(f"Saved selected-topic rank table to: {TOP_TOPIC_RANK_TABLE_EXPORT_PATH}")
print(f"Saved all-topic long keyword table to: {ALL_TOPIC_KEYWORDS_EXPORT_PATH}")
print(f"Saved all-topic rank table to: {ALL_TOPIC_RANK_TABLE_EXPORT_PATH}")


In [ ]:
APPROX_DISTRIBUTION_BATCH_SIZE = 256
APPROX_DISTRIBUTION_WINDOW = 4
APPROX_DISTRIBUTION_STRIDE = 1
APPROX_DISTRIBUTION_MIN_SIMILARITY = 0.1
APPROX_DISTRIBUTION_USE_EMBEDDINGS = False

# approximate_distribution(...) needs a live c-TF-IDF representation.
# If the saved merged model was refreshed for the 20-keyword exports above,
# `keyword_model` is the best object to reuse here.
distribution_model = keyword_model if "keyword_model" in locals() else merged_model
canonical_distribution_topic_info = (
    canonical_topic_info_named if "canonical_topic_info_named" in locals() else merged_topic_info_named
)

approx_distribution_df, approx_topic_column_reference_df, topic_distr = moa.build_approximate_distribution_dataset(
    distribution_model,
    article_topic_df,
    canonical_distribution_topic_info,
    batch_size=APPROX_DISTRIBUTION_BATCH_SIZE,
    window=APPROX_DISTRIBUTION_WINDOW,
    stride=APPROX_DISTRIBUTION_STRIDE,
    min_similarity=APPROX_DISTRIBUTION_MIN_SIMILARITY,
    use_embedding_model=APPROX_DISTRIBUTION_USE_EMBEDDINGS,
)

topic_probability_columns = approx_topic_column_reference_df["probability_column"].tolist()
approx_distribution_sums = approx_distribution_df[topic_probability_columns].sum(axis=1)

approx_top3_rows = []
for row_index, probs in enumerate(topic_distr):
    top_indices = probs.argsort()[::-1][:3]
    article_row = approx_distribution_df.iloc[row_index]
    top_entry = {
        "row_id": article_row.get("row_id", pd.NA),
        "Title": article_row.get("Title", pd.NA),
        "outlet_label": article_row.get("outlet_label", pd.NA),
        "assigned_topic": article_row.get("merged_topic", pd.NA),
        "assigned_label": article_row.get("topic_display_label", pd.NA),
    }
    for rank_position, topic_idx in enumerate(top_indices, start=1):
        topic_id = int(approx_topic_column_reference_df.iloc[topic_idx]["Topic"])
        topic_label = approx_topic_column_reference_df.iloc[topic_idx]["DisplayLabel"]
        top_entry[f"approx_topic_{rank_position}"] = topic_id
        top_entry[f"approx_label_{rank_position}"] = topic_label
        top_entry[f"approx_prob_{rank_position}"] = float(probs[topic_idx])
    approx_top3_rows.append(top_entry)

approx_top3_df = pd.DataFrame(approx_top3_rows)

display(approx_topic_column_reference_df.head(12))
display(
    approx_distribution_df[
        [
            col
            for col in [
                "row_id",
                "outlet_label",
                "Title",
                "merged_topic",
                "topic_display_label",
                "approx_topic",
                "approx_display_label",
                "approx_topic_probability",
            ]
            if col in approx_distribution_df.columns
        ]
    ].head(10)
)
display(approx_top3_df.head(10))

print(f"topic_distr shape: {topic_distr.shape}")
print(f"topic probability columns: {len(topic_probability_columns)}")
print(
    "Probability row sums (should be close to 1): "
    f"min={approx_distribution_sums.min():.4f}, "
    f"median={approx_distribution_sums.median():.4f}, "
    f"max={approx_distribution_sums.max():.4f}"
)
print(
    "approx_topic_probability is the highest approximate topic-mixture share per article. "
    "It is not the same as merged_probability from transform()."
)
